In [1]:
import pandas as pd

RAW_PATH = "../data/raw/accepted_2007_to_2018Q4.csv.gz"

sample = pd.read_csv(RAW_PATH, nrows=1000, low_memory=False)

print("Columns:", sample.shape[1])
sample.head()

Columns: 151


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
import time

RAW_PATH = "../data/raw/accepted_2007_to_2018Q4.csv.gz"
CHUNK_SIZE = 100_000

start = time.time()
total_rows = 0
max_len = None
null_counts = None

for chunk in pd.read_csv(RAW_PATH, chunksize=CHUNK_SIZE, dtype=str):
    total_rows += len(chunk)

    chunk_max = chunk.apply(lambda col: col.str.len().max())
    chunk_nulls = chunk.isna().sum()

    if max_len is None:
        max_len = chunk_max
        null_counts = chunk_nulls
    else:
        max_len = pd.concat([max_len, chunk_max], axis=1).max(axis=1)
        null_counts = null_counts + chunk_nulls

    print(f"Rows processed: {total_rows:,}", end="\r")

print(f"\nTotal rows: {total_rows:,}")
print(f"Time: {time.time() - start:.0f} seconds")

Rows processed: 2,260,701
Total rows: 2,260,701
Time: 47 seconds


In [3]:
profile = pd.DataFrame({
    "max_length": max_len,
    "null_count": null_counts,
})
profile["null_pct"] = (profile["null_count"] / total_rows * 100).round(1)

profile.to_csv("../data/processed/column_profile.csv")

profile.sort_values("max_length", ascending=False).head(15)

,max_length,null_count,null_pct
desc,3988.0,2134636,94.4
title,80.0,23359,1.0
emp_title,78.0,167002,7.4
url,66.0,33,0.0
loan_status,51.0,33,0.0
id,48.0,0,0.0
hardship_type,31.0,2249784,99.5
funded_amnt_inv,22.0,33,0.0
total_rec_late_fee,22.0,33,0.0
hardship_reason,21.0,2249784,99.5


In [4]:
import math
from pathlib import Path
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

connection_url = URL.create(
    "mssql+pyodbc",
    host="localhost",
    database="LoanDefault",
    query={
        "driver": "ODBC Driver 17 for SQL Server",
        "trusted_connection": "yes",
    },
)
engine = create_engine(connection_url, fast_executemany=True)


def nvarchar_size(max_length):
    if pd.isna(max_length):
        return 50
    size = math.ceil(max_length * 1.2 / 10) * 10
    return min(max(size, 10), 4000)


column_lines = [
    f"    [{col}] NVARCHAR({nvarchar_size(length)}) NULL"
    for col, length in profile["max_length"].items()
]

create_sql = (
    "DROP TABLE IF EXISTS dbo.raw_loans;\n\n"
    "CREATE TABLE dbo.raw_loans (\n"
    + ",\n".join(column_lines)
    + "\n);"
)

Path("../sql").mkdir(exist_ok=True)
Path("../sql/01_create_raw_loans.sql").write_text(create_sql)

print(create_sql[:700])

DROP TABLE IF EXISTS dbo.raw_loans;

CREATE TABLE dbo.raw_loans (
    [id] NVARCHAR(60) NULL,
    [member_id] NVARCHAR(50) NULL,
    [loan_amnt] NVARCHAR(10) NULL,
    [funded_amnt] NVARCHAR(10) NULL,
    [funded_amnt_inv] NVARCHAR(30) NULL,
    [term] NVARCHAR(20) NULL,
    [int_rate] NVARCHAR(10) NULL,
    [installment] NVARCHAR(10) NULL,
    [grade] NVARCHAR(10) NULL,
    [sub_grade] NVARCHAR(10) NULL,
    [emp_title] NVARCHAR(100) NULL,
    [emp_length] NVARCHAR(20) NULL,
    [home_ownership] NVARCHAR(10) NULL,
    [annual_inc] NVARCHAR(20) NULL,
    [verification_status] NVARCHAR(20) NULL,
    [issue_d] NVARCHAR(10) NULL,
    [loan_status] NVARCHAR(70) NULL,
    [pymnt_plan] NVARCHAR(10


In [5]:
with engine.begin() as conn:
    conn.execute(text(create_sql))

print("Table created.")

Table created.


In [6]:
start = time.time()
loaded = 0

for chunk in pd.read_csv(RAW_PATH, chunksize=CHUNK_SIZE, dtype=str):
    chunk.to_sql(
        "raw_loans",
        engine,
        schema="dbo",
        if_exists="append",
        index=False,
        chunksize=10_000,
    )
    loaded += len(chunk)
    elapsed_min = (time.time() - start) / 60
    print(f"Loaded {loaded:,} rows | {elapsed_min:.1f} min", end="\r")

print(f"\nDone: {loaded:,} rows in {(time.time() - start) / 60:.1f} minutes")

Loaded 2,260,701 rows | 7.0 min
Done: 2,260,701 rows in 7.0 minutes
